In [ ]:
import os
import time
import torch
import skimage
import sklearn.metrics

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

In [ ]:
import mnds
import extractor
import detection
import vision_transformer as vit

In [ ]:
PATCH_SIZE = 256
STRIDE = 8
FEATURE_SIZE = 384
TOKENS_PER_PATCH = PATCH_SIZE // STRIDE
DIRECTORY = "/home/caicedo/scr/jcaicedo/Micronuclei-data/"

BATCH_SIZE = 48
EPOCHS = 20
LR = 0.01

device = 'cuda:2' if torch.cuda.is_available() else 'cpu'

In [ ]:
filelist = os.listdir(DIRECTORY)
annot_files = [x for x in filelist if x.endswith('png')]

training_files = annot_files[0:-1]
validation_files = [annot_files[-1]]

In [ ]:
validation_set = mnds.MicronucleiDataset(filelist=validation_files, directory=DIRECTORY, mode="fixed")

In [ ]:
val_dataloader = DataLoader(validation_set, batch_size=4, shuffle=False)

In [ ]:
model_file = DIRECTORY + "models/" + validation_files[0].replace('phenotype_outlines.png','pth')
model = torch.load(model_file)
model

In [ ]:
def display_examples(vin, vls, pred):
    for j in range(pred.shape[0]):
        # Visualize predictions
        #plt.figure(figsize=(9,3))
        fig, ax = plt.subplots(1,4)
        
        # Input image
        ax[0].imshow(vin[j][0,...])
        ax[0].axis('off')

        # Ground truth
        ax[1].imshow(vls[j])
        ax[1].axis('off')

        # Detections
        ax[2].imshow(pred[j] > 0.1)
        ax[2].axis('off')

        # Probability map
        ax[3].imshow(pred[j] )
        ax[3].axis('off')

        plt.show()


In [ ]:
model.eval()

GT = []
PRED = []

with torch.no_grad():
    for i, vdata in enumerate(val_dataloader):
        # Get predictions
        vin, vls = vdata
        pred0 = model(vin.to(device))
        P = torch.reshape(pred0, (-1, 32, 32))
        pred = P.cpu().numpy()
        
        # Collect predictions and ground truth
        PRED.append(pred)
        GT.append(vls.cpu().numpy())
        
        #if i % 20 == 0: 
        display_examples(vin, vls, pred)
            
PRED = np.concatenate(PRED, axis=0).reshape((-1,))
GT = np.concatenate(GT, axis=0).reshape((-1,))

In [ ]:
# Precision-recall curve
display = sklearn.metrics.PrecisionRecallDisplay.from_predictions(
    GT, PRED, name="Detector", plot_chance_level=True
)
_ = display.ax_.set_title("Precision-Recall curve")

In [ ]:
# Classification report
report = sklearn.metrics.classification_report(GT, PRED > 0.1)
print(report)

# NOTES FOR NEXT STEPS:
1. We do not need a dataloader for prediction. 
2. We can independently load the image and pass patches individually as we wish.
3. Let's scan the image with a stride that overlaps patches of 256x256.
4. Then, collect predictions in a buffer with the same size of the original image.
5. Given that we will make predictions with overlapping patches, let's accumulate the predictions.
6. To take the average, simply divide by the number of predictions in a token.
7. This can be tracked in another buffer that accumulates ones after predictions are obtained.
8. The ground truth is in a DataFrame. Use it to decide true positives and so on.
9. Generate visualizations of the predictions and status w.r.t. ground truth.
